[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pachterlab/OP_2025_2/blob/main/analysis_scripts/Method_Overview.ipynb)

# accumap, step by step (Fig. 1)

This notebook builds **Fig. 1** of the paper: an illustration of accumap's reconstruction on a small toy array. It needs no data and no GPU and runs in a minute or two.

**What is computed and what is constructed.** The toy array, its barcode counts, the sender placement, the kernel fit, the density map, a bead's likelihood surface and the contact-direction tensors are all computed. The UMAP layout's distortion (one region compressed, another sheared) and its step-by-step removal are *constructed*, so that each step of the method shows one effect clearly. The real method's behavior on real data is in `Experimental_Ground_Truth.ipynb`.

In [ ]:
# Colab only: clone the repository (main branch) and install accumap; skipped when run locally
import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("/content/OP_2025_2"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main", "https://github.com/pachterlab/OP_2025_2.git", "/content/OP_2025_2"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/OP_2025_2"], check=True)
if "google.colab" in sys.modules:
    os.chdir("/content/OP_2025_2/analysis_scripts")

In [ ]:
import os
import colorsys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, FancyArrowPatch
from scipy.ndimage import gaussian_filter
import sys

FIGURES = '../final_figures'   # where the paper's figures are written
os.makedirs(FIGURES, exist_ok=True)
TITLE_FS, LABEL_FS, TICK_FS, LETTER_FS = 12, 11, 9, 16
rng = np.random.default_rng(3)

## 1. A toy array and its counts

Beads are packed in a disc and split at random into receivers and senders. Counts are drawn from a Poisson model with a contact-dominated kernel, as in accumap's count model (main-text Methods).

In [ ]:
# ------------------------------------------------------------------ toy truth and counts
def packed_disc(spacing=0.026, jitter=0.0065, r=1.0):
    pts = []
    for i, y in enumerate(np.arange(-r, r + spacing, spacing * np.sqrt(3) / 2)):
        for x in np.arange(-r, r + spacing, spacing) + (spacing / 2 if i % 2 else 0):
            if np.hypot(x, y) <= r:
                pts.append([x, y])
    P = np.array(pts) + rng.normal(0, jitter, (len(pts), 2))
    return P[rng.permutation(len(P))]


Bd = packed_disc()
R, S = Bd[: len(Bd) // 2], Bd[len(Bd) // 2:]   # receivers, senders
SPACING = 0.036
K = lambda d: np.exp(-(d / SPACING) ** 2) + 0.0003
D = np.linalg.norm(R[:, None] - S[None], axis=2)
C = rng.poisson(40 * K(D))                       # counts (receivers x senders)

## 2. A distorted "UMAP" layout

UMAP's error on real arrays is mostly a smooth warp (Supp Fig S3). The toy layout is the truth compressed on one side (a distortion the **uniform-density prior** detects) and sheared on the other (one the **contact-isotropy prior** detects), plus bead-level jitter. The intermediate layouts below remove these components in the order the method addresses them.

In [ ]:
def compress(P):                                 # smooth radial compression of the right half
    x, y = P[:, 0], P[:, 1]
    f = 1 - 0.38 * np.exp(-((x - 0.45) ** 2 + y ** 2) / 0.22)
    c = np.array([0.45, 0.0])
    return c + (P - c) * f[:, None]


def shear(P):                                    # smooth area-preserving shear of the left half
    x, y = P[:, 0], P[:, 1]
    w = np.exp(-((x + 0.45) ** 2 + y ** 2) / 0.2)
    return np.c_[x + 0.9 * w * y, y]


jit = rng.normal(0, 0.025, R.shape)
L_umap = compress(shear(R)) + jit                # 'UMAP' layout of the receivers
L_warp = shear(R) + 0.5 * jit                    # after warp with density prior: compression removed
L_ref = shear(R) + 0.15 * jit                    # after per-bead refinement: jitter reduced
L_final = R + 0.15 * jit                         # after isotropy stage: shear removed
jitS = rng.normal(0, 0.025, S.shape)             # senders move with the warp, as in the method
S_ref, S_final = shear(S) + 0.15 * jitS, S + 0.15 * jitS

hue = 0.05 + 0.85 * (R[:, 0] + 1) / 2
light = 0.3 + 0.45 * (R[:, 1] + 1) / 2
COL = np.array([colorsys.hls_to_rgb(h, l, 0.7) for h, l in zip(hue, light)])

## 3. Sender placement and plotting helpers

Each sender starts at the count-weighted median of its receivers (panel C).

In [ ]:
def senders_at_median(L):
    U = np.zeros((len(S), 2))
    for o in range(len(S)):
        w = C[:, o].astype(float)
        if w.sum() == 0:
            U[o] = np.nan; continue
        for k in range(2):
            idx = np.argsort(L[:, k]); cw = np.cumsum(w[idx])
            U[o, k] = L[idx[np.searchsorted(cw, cw[-1] / 2)], k]
    return U


U_umap = senders_at_median(L_umap)


def beads(ax, L, s=5, alpha=1.0):
    ax.scatter(L[:, 0], L[:, 1], c=COL, s=s, linewidths=0, alpha=alpha, zorder=3)


def grid(ax, f, color="#1a1a1a", lw=0.9):
    """Regular grid of the true layout mapped through f: shows compression and shear at a glance."""
    t = np.linspace(-1, 1, 200)
    for c in np.arange(-0.8, 0.81, 0.4):
        for P in (np.c_[np.full_like(t, c), t], np.c_[t, np.full_like(t, c)]):
            P = P[np.hypot(P[:, 0], P[:, 1]) <= 0.98]
            Q = f(P)
            ax.plot(Q[:, 0], Q[:, 1], color=color, lw=lw, zorder=4)


def frame(ax, title, letter, lim=1.2):
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_title(title, fontsize=TITLE_FS)
    ax.text(-0.02, 1.16, letter, transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")


def contact_ellipses(ax, L, U, color):
    for cx in (-0.6, 0.0, 0.6):
        for cy in (-0.6, 0.0, 0.6):
            if np.hypot(cx, cy) > 0.9:
                continue
            V = []
            for r_ in np.nonzero(np.linalg.norm(L - [cx, cy], axis=1) < 0.42)[0]:
                for o in np.argsort(C[r_])[::-1][:3]:
                    if C[r_, o] >= 2 and np.isfinite(U[o]).all():
                        v = L[r_] - U[o]; n = np.hypot(*v)
                        if 1e-6 < n < 0.15:
                            V.append(v / n)
            V = np.array(V); T = V.T @ V / len(V)
            w, E = np.linalg.eigh(T)
            ratio = min((w[1] / w[0]) ** 2, 3.0)            # anisotropy exaggerated (squared) for display
            a = 0.36 * np.sqrt(ratio); bb = 0.36 / np.sqrt(ratio)
            ang = np.degrees(np.arctan2(E[1, 1], E[0, 1]))
            ax.add_patch(Ellipse((cx, cy), a, bb, angle=ang, facecolor="none", edgecolor=color, lw=2.0, zorder=5))
            # long axis, drawn with length proportional to the elongation (zero for a circle)
            half = 0.5 * a * min(1.0, (a / bb - 1.0) / 0.6)
            if half > 0.01:
                t = np.radians(ang)
                ax.plot([cx - half * np.cos(t), cx + half * np.cos(t)], [cy - half * np.sin(t), cy + half * np.sin(t)],
                        ls="--", color=color, lw=2.0, zorder=6)

## 4. The figure

Panels: (A) barcode transfer; (B) the UMAP layout; (C) sender placement; (D) the kernel fit on the UMAP layout; (E) the smooth warp with the density prior; (F, G) per-bead refinement of receivers, then senders; (H) contact directions before the second stage; (I) the final layout.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13.5, 14.5))
axes = axes.ravel()
ident = lambda P: P

# A: barcode transfer at the truth
ax = axes[0]
b = np.argmin(np.linalg.norm(R - [0.05, 0.1], axis=1))
z = 0.12
win = np.all(np.abs(S - R[b]) < z, axis=1); winr = np.all(np.abs(R - R[b]) < z, axis=1)
ax.scatter(S[win, 0], S[win, 1], s=46, color="#1a1a1a", linewidths=0, zorder=2)          # senders: black
ax.scatter(R[winr, 0], R[winr, 1], s=46, color="#eb6834", linewidths=0, alpha=0.55, zorder=2)  # receivers: orange
for o in np.argsort(C[b])[::-1][:6]:
    ax.add_patch(FancyArrowPatch(S[o], R[b], arrowstyle="-|>", mutation_scale=10, lw=0.5 + 0.06 * C[b, o],
                                 color="#6f6e6b", zorder=4, shrinkA=4, shrinkB=6))
ax.scatter(*R[b], s=110, color="#eb6834", edgecolors="k", linewidths=1.4, zorder=5)
ax.set_xlim(R[b, 0] - z, R[b, 0] + z); ax.set_ylim(R[b, 1] - z, R[b, 1] + z)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Senders (black) transfer barcodes to\nthe receivers (orange) they touch: counts $C$", fontsize=TITLE_FS)
ax.text(-0.02, 1.16, "A", transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")

# B: UMAP embedding (grid: the truth's regular grid carried along)
ax = axes[1]
beads(ax, L_umap, alpha=0.8); grid(ax, lambda P: compress(shear(P)))
frame(ax, "UMAP embedding of receivers:\nright compressed, left sheared", "B")

# C: sender placement
ax = axes[2]
ok = np.isfinite(U_umap).all(1)
cand = [o for o in np.argsort(np.linalg.norm(np.nan_to_num(U_umap, nan=9) - [-0.1, -0.3], axis=1))[:40] if (C[:, o] >= 2).sum() >= 6]
o = cand[0]; zc = 0.12; c0 = U_umap[o]
inw = lambda P: np.all(np.abs(P - c0) < zc * 1.05, axis=1)
ax.scatter(L_umap[inw(L_umap), 0], L_umap[inw(L_umap), 1], s=40, color="#eb6834", linewidths=0, alpha=0.35, zorder=2)
oth = inw(np.nan_to_num(U_umap, nan=9)) & ok; oth[o] = False
ax.scatter(U_umap[oth, 0], U_umap[oth, 1], s=40, color="#1a1a1a", linewidths=0, alpha=0.35, zorder=2)
for r_ in np.nonzero(C[:, o] >= 2)[0]:
    ax.plot([c0[0], L_umap[r_, 0]], [c0[1], L_umap[r_, 1]], color="#6f6e6b", lw=0.5 + 0.12 * C[r_, o], zorder=3)
    ax.scatter(*L_umap[r_], s=60, color="#eb6834", edgecolors="k", linewidths=0.8, zorder=4)
ax.scatter(*c0, s=130, color="#1a1a1a", edgecolors="white", linewidths=1.5, zorder=5)
ax.set_xlim(c0[0] - zc, c0[0] + zc); ax.set_ylim(c0[1] - zc, c0[1] + zc)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Each sender placed at the count-weighted\nmedian of its receivers (lines: counts)", fontsize=TITLE_FS)
ax.text(-0.02, 1.16, "C", transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")

# D: kernel fit on the current layout
ax = axes[3]
Du = np.linalg.norm(L_umap[:, None] - U_umap[None], axis=2)[:, ok]; Cu = C[:, ok]
bins = np.linspace(0, 0.3, 25); mid = 0.5 * (bins[1:] + bins[:-1])
mean_c = np.array([Cu[(Du >= a) & (Du < b_)].mean() for a, b_ in zip(bins[:-1], bins[1:])])
ax.plot(mid, mean_c, "o", color="#4169e1", ms=5, mec="white", label="mean count, UMAP layout", zorder=3)
from scipy.optimize import curve_fit
popt, _ = curve_fit(lambda d, A, s0, c0: A * np.exp(-(d / s0) ** 2) + c0, mid, mean_c, p0=(20, 0.05, 0.0))
dd = np.linspace(0, 0.3, 200)
ax.plot(dd, popt[0] * np.exp(-(dd / popt[1]) ** 2) + popt[2], color="#4169e1", lw=2, label="fitted kernel")
ax.plot(dd, 40 * K(dd), color="#8a8986", lw=1.5, ls="--", label="true kernel")
ax.set_xlabel("distance in layout", fontsize=LABEL_FS); ax.set_ylabel("count", fontsize=LABEL_FS)
ax.set_xticks([]); ax.set_yticks([])
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.legend(fontsize=TICK_FS, frameon=False)
ax.set_title("Fit the transfer kernel $K(d)$ and per-\nbead parameters to $C$ on this layout", fontsize=TITLE_FS)
ax.text(-0.12, 1.16, "D", transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")

# E: warp with density prior (zoom on the compressed region)
ax = axes[4]
ce, ze = np.array([0.45, 0.0]), 0.5
g = np.linspace(-1.2, 1.2, 240)
H, _, _ = np.histogram2d(L_umap[:, 1], L_umap[:, 0], bins=[g, g])
ax.imshow(gaussian_filter(H, 12), extent=(-1.2, 1.2, -1.2, 1.2), origin="lower", cmap="Greys", alpha=0.6, zorder=1)
beads(ax, L_umap, s=9, alpha=0.6)
grid(ax, lambda P: compress(shear(P)), color="#6f6e6b", lw=0.8)
gx = np.arange(-1, 1.01, 0.18); G = np.array([[x, y] for x in gx for y in gx if np.hypot(x, y) < 0.95])
src = compress(shear(G)); disp = shear(G) - src
for p_, v in zip(src, disp):
    if np.hypot(*v) > 0.01 and np.all(np.abs(p_ - ce) < ze):
        ax.add_patch(FancyArrowPatch(p_, p_ + v, arrowstyle="-|>", mutation_scale=11, lw=1.6, color="#dc143c", zorder=5))
ax.set_xlim(ce[0] - ze, ce[0] + ze); ax.set_ylim(ce[1] - ze, ce[1] + ze)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Smooth warp (arrows): likelihood +\nuniform-density prior (gray: density)", fontsize=TITLE_FS)
ax.text(-0.02, 1.16, "E", transform=ax.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")

# F: per-bead refinement, alternating receivers (i) and senders (ii)
axL, axR = axes[5], axes[6]
U_w = senders_at_median(L_warp); okw = np.isfinite(U_w).all(1)
h = 0.05; gg = np.linspace(-h, h, 61)


def refine_panel(ax, centre, others, counts, mover_style, other_style, sub_title):
    """Likelihood of the moving bead over a +-h grid, its partners and its move to the maximum."""
    ll = np.zeros((61, 61))
    for i_, dy in enumerate(gg):
        for j_, dx in enumerate(gg):
            mu = 40 * K(np.linalg.norm(others - (centre + [dx, dy]), axis=1))
            ll[i_, j_] = (counts * np.log(mu) - mu).sum()
    ext = (centre[0] - h, centre[0] + h, centre[1] - h, centre[1] + h)
    ax.imshow(ll - ll.max(), extent=ext, origin="lower", cmap="Blues", vmin=np.percentile(ll - ll.max(), 10), vmax=0, zorder=1)
    near = np.all(np.abs(others - centre) < h, axis=1)
    ax.scatter(others[near & (counts < 2), 0], others[near & (counts < 2), 1], s=30, color=other_style["faint"], linewidths=0,
               alpha=0.6, zorder=3)
    ax.scatter(others[counts >= 2, 0], others[counts >= 2, 1], s=45, color=other_style["color"], edgecolors=other_style["edge"],
               linewidths=1.0, zorder=3)
    i_, j_ = np.unravel_index(ll.argmax(), ll.shape)
    ax.scatter(*centre, s=80, color=mover_style["color"], edgecolors=mover_style["edge"], linewidths=1.2, zorder=4)
    ax.add_patch(FancyArrowPatch(centre, centre + [gg[j_], gg[i_]], arrowstyle="-|>", mutation_scale=11, lw=1.7, color="#dc143c", zorder=5))
    ax.set_xlim(ext[:2]); ax.set_ylim(ext[2:]); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(sub_title, fontsize=TITLE_FS)


ORANGE = {"color": "#eb6834", "edge": "k", "faint": "#eb6834"}
BLACK = {"color": "#1a1a1a", "edge": "white", "faint": "#8a8986"}
b = np.argmin(np.linalg.norm(L_warp - [0.3, 0.3], axis=1))
refine_panel(axL, L_warp[b], U_w[okw], C[b, okw], ORANGE, BLACK, "Per-bead refinement: each receiver\n(orange) to its likelihood maximum (blue)")
cand = [o for o in np.argsort(np.linalg.norm(np.nan_to_num(U_w, nan=9) - [0.3, 0.3], axis=1))[:30] if (C[:, o] >= 2).sum() >= 3]
o = cand[0]
refine_panel(axR, U_w[o], L_warp, C[:, o], BLACK, ORANGE, "... then each sender (black), with the\nreceivers fixed; repeated until converged")
axL.text(-0.02, 1.16, "F", transform=axL.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")
axR.text(-0.02, 1.16, "G", transform=axR.transAxes, fontsize=LETTER_FS, fontweight="bold", va="bottom")

# H: second stage, before
ax = axes[7]
beads(ax, L_ref, alpha=0.45); grid(ax, shear, color="#6f6e6b", lw=0.7)
contact_ellipses(ax, L_ref, S_ref, "#dc143c")
frame(ax, "Second stage: contact directions should\nbe isotropic (dashes: elongation)", "H")

# I: result
ax = axes[8]
beads(ax, L_final, alpha=0.45); grid(ax, ident, color="#6f6e6b", lw=0.7)
contact_ellipses(ax, L_final, S_final, "#4169e1")
frame(ax, "Isotropy warp and second refinement:\nfinal layout", "I")

fig.subplots_adjust(wspace=0.15, hspace=0.38)
fig.savefig(os.path.join(FIGURES, "Fig1_final.png"), dpi=300, bbox_inches="tight")
plt.show()